In [ ]:
# =========================
# 0) Install + Imports
# =========================
!pip -q install scikit-learn openpyxl pandas numpy joblib

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

# ============================================================
# 1) Settings
# ============================================================
TRAIN_PATH = "/content/train.xlsx"
VALID_PATH = "/content/valid.xlsx"
TEST_PATH  = "/content/test.xlsx"

TEXT_COL = "clean_text"
LABELS = ["dialect", "hate", "topic"]

OUTPUT_DIR = "./logistic_multilabel_model"

TARGET_NAMES = {
    "dialect": ["مصري", "سعودي"],
    "hate": ["غير كراهية", "كراهية"],
    "topic": ["سياسي", "ديني"]
}

id2label = {
    "dialect": {0: "مصري", 1: "سعودي"},
    "hate": {0: "غير كراهية", 1: "كراهية"},
    "topic": {0: "سياسي", 1: "ديني"}
}

# ============================================================
# 2) Emoji Feature
# ============================================================
HATE_EMOJIS = set([
    '💦','🐖','🐷','🐽','👞','🐕','🐶','💩','🐄','🐮',
    '🐑','🐏','👎','😡','🤬','👺','👿','😠'
])

def emoji_flag(text):
    text = "" if pd.isna(text) else str(text)
    return 1 if any(e in text for e in HATE_EMOJIS) else 0

class EmojiFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.Series):
            features = X.apply(lambda x: emoji_flag(x))
        else:
            features = pd.Series(X).apply(lambda x: emoji_flag(x))
        return features.values.reshape(-1, 1)

# ============================================================
# 3) Load Data - Clean Text Already Preprocessed
# ============================================================
def load_data(path):
    df = pd.read_excel(path, engine="openpyxl")
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()
    return df

print("📂 Loading data...")
train_df = load_data(TRAIN_PATH)
valid_df = load_data(VALID_PATH)
test_df  = load_data(TEST_PATH)

# ============================================================
# 4) Prepare
# ============================================================
X_train = train_df[[TEXT_COL]]
X_valid = valid_df[[TEXT_COL]]
X_test  = test_df[[TEXT_COL]]

y_train = train_df[LABELS].values.astype(int)
y_valid = valid_df[LABELS].values.astype(int)
y_test  = test_df[LABELS].values.astype(int)

# ============================================================
# 5) Model
# ============================================================
features_pipeline = ColumnTransformer([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2)), TEXT_COL),
    ("emoji", EmojiFeatureExtractor(), TEXT_COL)
])

model = Pipeline([
    ("features", features_pipeline),
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")
    ))
])

# ============================================================
# 6) Train
# ============================================================
print("\n⏳ Training...")
model.fit(X_train, y_train)

# ============================================================
# 7) Evaluation
# ============================================================
def evaluate(y_true, y_pred, name):
    print(f"\n===== {name} =====")
    for i, label in enumerate(LABELS):
        print(f"\n--- {label} ---")
        print(classification_report(
            y_true[:, i],
            y_pred[:, i],
            labels=[0, 1],
            target_names=TARGET_NAMES[label],
            digits=4,
            zero_division=0
        ))

y_valid_pred = model.predict(X_valid)
y_test_pred  = model.predict(X_test)

evaluate(y_valid, y_valid_pred, "VALIDATION")
evaluate(y_test, y_test_pred, "TEST")

📂 Loading data...

⏳ Training...

===== VALIDATION =====

--- dialect ---
              precision    recall  f1-score   support

        مصري     0.9203    0.8467    0.8819       300
       سعودي     0.8558    0.9254    0.8893       295

    accuracy                         0.8857       595
   macro avg     0.8880    0.8860    0.8856       595
weighted avg     0.8883    0.8857    0.8856       595


--- hate ---
              precision    recall  f1-score   support

  غير كراهية     0.7287    0.7884    0.7574       293
      كراهية     0.7770    0.7152    0.7448       302

    accuracy                         0.7513       595
   macro avg     0.7528    0.7518    0.7511       595
weighted avg     0.7532    0.7513    0.7510       595


--- topic ---
              precision    recall  f1-score   support

       سياسي     0.9676    0.9934    0.9803       301
        ديني     0.9930    0.9660    0.9793       294

    accuracy                         0.9798       595
   macro avg     0.9803  